# LA Studio Voice Cloning - OmniVoice

This notebook loads exactly `omnivoice` (`k2-fsa/OmniVoice`) on CUDA.
It is independent from API Gateway and refuses every other model ID.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's Voice Cloning panel.


In [ ]:
!nvidia-smi
%pip install -q "git+https://github.com/k2-fsa/OmniVoice.git@468e927ba371" "soundfile==0.13.1" "python-multipart==0.0.20" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_clone_worker.py')
WORKER.write_text('import torch\n\nfrom omnivoice import OmniVoice\n\nMODEL_ID = "omnivoice"\nMODEL_NAME = "OmniVoice"\nUPSTREAM_MODEL = "k2-fsa/OmniVoice"\nMODEL = OmniVoice.from_pretrained(UPSTREAM_MODEL, device_map="cuda:0", dtype=torch.float16)\n\ndef prepare_exact_profile(profile):\n    return MODEL.create_voice_clone_prompt(\n        ref_audio=profile["ref_audio"],\n        ref_text=profile["ref_text"],\n    )\n\ndef clone_with_exact_model(profile, request):\n    audio = MODEL.generate(\n        text=request.text,\n        voice_clone_prompt=profile["state"],\n        speed=request.speed,\n        num_step=request.num_step,\n    )\n    return audio, 24000\n\nimport io\nimport os\nimport shutil\nimport tempfile\nimport threading\nimport uuid\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_VOICE_CLONE_TOKEN"]\nDATA_DIR = Path("/content/la-studio-voice-clone-data") / MODEL_ID\nDATA_DIR.mkdir(parents=True, exist_ok=True)\nMAX_REFERENCE_BYTES = 256 * 1024 * 1024\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nMODEL_LOCK = threading.Lock()\nSTATE_LOCK = threading.Lock()\nPROFILES = {}\nJOBS = {}\n\nclass GenerationRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    profile_id: str = Field(min_length=1, max_length=160)\n    text: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    language: str = Field(default="vi", max_length=40)\n    speed: float = Field(default=1.0, ge=0.1, le=2.0)\n    num_step: int = Field(default=32, ge=1, le=64)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef require_exact_model(model: str) -> None:\n    if model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{model}\'. Open the notebook for the selected model.",\n        )\n\ndef audio_array(value):\n    if isinstance(value, (list, tuple)):\n        if not value:\n            raise RuntimeError("the selected model returned no audio")\n        value = value[0]\n    if torch.is_tensor(value):\n        value = value.detach().float().cpu().numpy()\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\n    if audio.size == 0 or not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned invalid audio")\n    return audio\n\ndef write_wav(path: Path, value, sample_rate: int) -> None:\n    audio = audio_array(value)\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise RuntimeError("generated audio exceeds the five minute output limit")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    sf.write(path, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n\ndef public_job(job_id: str):\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if not job:\n            raise HTTPException(status_code=404, detail="voice job not found")\n        return {key: value for key, value in job.items() if key not in {"audio_path", "cancelled"}}\n\ndef fail_job(job_id: str, error: Exception) -> None:\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if job:\n            job.update({\n                "status": "failed",\n                "stage": "failed",\n                "error": {"message": f"{type(error).__name__}: {str(error)[:300]}"},\n            })\n\ndef build_profile(job_id: str, profile_id: str) -> None:\n    try:\n        with STATE_LOCK:\n            profile = PROFILES[profile_id]\n            JOBS[job_id].update({"status": "running", "stage": "prepare_profile", "percent": 10})\n        with MODEL_LOCK:\n            state = prepare_exact_profile(profile)\n        with STATE_LOCK:\n            profile["state"] = state\n            JOBS[job_id].update({\n                "status": "succeeded",\n                "stage": "complete",\n                "percent": 100,\n                "result": {"id": profile_id, "model": MODEL_ID},\n            })\n    except Exception as error:\n        fail_job(job_id, error)\n\ndef generate_audio(job_id: str, request: GenerationRequest) -> None:\n    try:\n        with STATE_LOCK:\n            profile = PROFILES.get(request.profile_id)\n            if not profile:\n                raise RuntimeError("voice profile no longer exists")\n            JOBS[job_id].update({"status": "running", "stage": "generate", "percent": 10})\n        with MODEL_LOCK:\n            audio, sample_rate = clone_with_exact_model(profile, request)\n        output_path = DATA_DIR / f"{job_id}.wav"\n        write_wav(output_path, audio, sample_rate)\n        with STATE_LOCK:\n            if JOBS[job_id].get("cancelled"):\n                JOBS[job_id].update({"status": "cancelled", "stage": "cancelled", "percent": 0})\n                output_path.unlink(missing_ok=True)\n            else:\n                JOBS[job_id].update({\n                    "status": "succeeded",\n                    "stage": "complete",\n                    "percent": 100,\n                    "audio_path": str(output_path),\n                    "result": {"model": MODEL_ID, "sample_rate": int(sample_rate)},\n                })\n    except Exception as error:\n        fail_job(job_id, error)\n\napp = FastAPI(title=f"LA Studio Voice Cloning - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-cloning",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "formats": ["wav"],\n                "reference_formats": ["wav", "mp3", "flac"],\n                "reference_duration_seconds": {"min": 3, "max": 30},\n                "requires_consent": True,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v2/jobs/profile", status_code=202)\nasync def create_profile(\n    model: str = Form(...),\n    name: str = Form(...),\n    consent_confirmed: bool = Form(...),\n    ref_text: str = Form(...),\n    language: str = Form(default="vi"),\n    separate_music: bool = Form(default=False),\n    ref_audio: UploadFile = File(...),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    if not consent_confirmed:\n        raise HTTPException(status_code=403, detail="explicit voice-cloning consent is required")\n    if not name.strip() or not ref_text.strip():\n        raise HTTPException(status_code=422, detail="profile name and exact reference transcript are required")\n    suffix = Path(ref_audio.filename or "").suffix.lower()\n    if suffix not in {".wav", ".mp3", ".flac"}:\n        raise HTTPException(status_code=415, detail="reference audio must be WAV, MP3, or FLAC")\n    profile_id = uuid.uuid4().hex\n    reference_path = DATA_DIR / f"{profile_id}{suffix}"\n    size = 0\n    with reference_path.open("wb") as output:\n        while chunk := await ref_audio.read(1024 * 1024):\n            size += len(chunk)\n            if size > MAX_REFERENCE_BYTES:\n                reference_path.unlink(missing_ok=True)\n                raise HTTPException(status_code=413, detail="reference audio exceeds 256 MB")\n            output.write(chunk)\n    try:\n        info = sf.info(reference_path)\n        duration = float(info.frames) / float(info.samplerate)\n    except Exception as error:\n        reference_path.unlink(missing_ok=True)\n        raise HTTPException(status_code=422, detail=f"reference audio cannot be decoded: {error}") from error\n    if duration < 3.0 or duration > 30.0:\n        reference_path.unlink(missing_ok=True)\n        raise HTTPException(status_code=422, detail="reference audio must be between 3 and 30 seconds")\n    job_id = uuid.uuid4().hex\n    profile = {\n        "id": profile_id,\n        "model": MODEL_ID,\n        "name": name.strip(),\n        "ref_audio": str(reference_path),\n        "ref_text": ref_text.strip(),\n        "language": language.strip() or "vi",\n        "separate_music": bool(separate_music),\n    }\n    with STATE_LOCK:\n        PROFILES[profile_id] = profile\n        JOBS[job_id] = {"id": job_id, "status": "queued", "stage": "queued", "percent": 0}\n    threading.Thread(target=build_profile, args=(job_id, profile_id), daemon=True).start()\n    return public_job(job_id)\n\n@app.post("/v2/jobs/generation", status_code=202)\ndef create_generation(request: GenerationRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    require_exact_model(request.model)\n    with STATE_LOCK:\n        profile = PROFILES.get(request.profile_id)\n        if not profile:\n            raise HTTPException(status_code=404, detail="voice profile not found")\n        if profile["model"] != MODEL_ID:\n            raise HTTPException(status_code=409, detail="voice profile belongs to a different model worker")\n        job_id = uuid.uuid4().hex\n        JOBS[job_id] = {"id": job_id, "status": "queued", "stage": "queued", "percent": 0}\n    threading.Thread(target=generate_audio, args=(job_id, request), daemon=True).start()\n    return public_job(job_id)\n\n@app.get("/v2/jobs/{job_id}/audio")\ndef job_audio(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        path = Path(job.get("audio_path", "")) if job else None\n    if not job:\n        raise HTTPException(status_code=404, detail="voice job not found")\n    if job.get("status") != "succeeded" or not path or not path.is_file():\n        raise HTTPException(status_code=409, detail="voice job audio is not ready")\n    return Response(path.read_bytes(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\n@app.get("/v2/jobs/{job_id}")\ndef job_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return public_job(job_id)\n\n@app.delete("/v2/jobs/{job_id}")\ndef cancel_job(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        job = JOBS.get(job_id)\n        if not job:\n            raise HTTPException(status_code=404, detail="voice job not found")\n        job["cancelled"] = True\n        if job["status"] == "queued":\n            job.update({"status": "cancelled", "stage": "cancelled", "percent": 0})\n    return {"cancelled": True}\n\n@app.delete("/v1/profiles/{profile_id}")\ndef delete_profile(profile_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with STATE_LOCK:\n        profile = PROFILES.pop(profile_id, None)\n    if profile:\n        Path(profile["ref_audio"]).unlink(missing_ok=True)\n    return {"deleted": bool(profile)}\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
import json, os, re, secrets, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

MODEL_ID = 'omnivoice'
TOKEN = secrets.token_urlsafe(32)
STARTUP_TIMEOUT_SECONDS = 20 * 60
WORKER_LOG = Path("/content/la_studio_voice_clone_worker.log")
env = os.environ.copy()
env["LA_STUDIO_COLAB_VOICE_CLONE_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"

def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"

def fail_startup(message: str) -> None:
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker.kill()
    raise RuntimeError(
        message + "\\n\\n---- LA Studio worker log (last 12,000 characters) ----\\n" + worker_log_tail()
    )

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "la_studio_voice_clone_worker:app", "--host", "127.0.0.1", "--port", "3923"],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print("Starting exact CUDA worker; initial model download can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            fail_startup(f"The exact-model worker exited before becoming ready (exit code {exit_code}).")
        try:
            check = urllib.request.Request(
                "http://127.0.0.1:3923/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(check, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and health.get("device") == "cuda"
                    and health.get("model") == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if int(time.monotonic()) % 30 == 0:
            print("Waiting for the exact CUDA model…", last_error)
        time.sleep(2)
    else:
        fail_startup(
            f"The exact-model worker did not become CUDA-ready within {STARTUP_TIMEOUT_SECONDS // 60} minutes. "
            f"Last health-check result: {last_error}"
        )

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3923", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_VOICE_CLONE_URL=" + public_url)
print("LA_STUDIO_COLAB_VOICE_CLONE_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_VOICE_CLONE_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
